# Plant thylakoid membrane formation and diffusion analysis

In [ ]:
import itertools
import logging
import os
import random
from copy import deepcopy
from multiprocessing import Pool
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from shapely import to_wkt, wkt
from shapely.affinity import rotate, translate
from shapely.geometry import MultiPolygon, Point, Polygon, box
from shapely.ops import unary_union
from shapely.strtree import STRtree
from tqdm import tqdm

import cyanomembranes as cm

(OUT := Path("output")).mkdir(exist_ok=True)

# Random generator
RNG = np.random.default_rng(seed=0)

# CPUS
N_PROCESSES = 10 #6

## Create protein shadows

In [ ]:
PDB_FILES = [
    "3JCU-PSII-spinach.pdb",
    "6RQF-Cyt-spinach.pdb",
    "7E0K-LHCII-chlamy.pdb"
]

DATA = Path("data_plants/")

proteins = cm.pdb_utils.process_proteins(DATA, OUT / "protein_shadows", PDB_FILES)


## Utilities

In [ ]:
def plot_membrane(filepath, ax):
    p = cm.geo_utils.readwkt(filepath)
    psi_area = proteins["3JCU-PSII-spinach"]["polygon"][0].area
    psii_area = proteins["6RQF-Cyt-spinach"]["polygon"][0].area
    cytb6f_area = proteins["7E0K-LHCII-chlamy"]["polygon"][0].area

    pastel_blue = "#A2CFFE"
    pastel_yellow = "#F7E1B5"
    pastel_green = "#A9E5B8"
    pastel_darkblue = "#54A5FC"

    psii_lst = []
    for i in p:
        if np.isclose(i.area, psi_area):
            ax.fill(*i.exterior.xy, lw=0.4, c=pastel_blue, edgecolor="grey")
        if np.isclose(i.area, psii_area):
            ax.fill(*i.exterior.xy, lw=0.4, c=pastel_yellow, edgecolor="grey")
            psii_lst.append(i)
        if np.isclose(i.area, cytb6f_area):
            ax.fill(*i.exterior.xy, lw=0.4, c="red", edgecolor="grey")



    ax.set_aspect("equal")
    ax.set_xlim(0,5000)
    ax.set_ylim(0,5000)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(8, 8))
ax = ax.flatten()
for i, prot in enumerate(proteins):
    polygon = proteins[prot]["polygon"]
    ax[i].plot(*polygon[0].exterior.xy)
    ax[i].set_aspect("equal")
    ax[i].set_title(prot)
    ax[i].set_xlim(-150, 150)
    ax[i].set_ylim(-150, 150)
plt.tight_layout()
fig.show()

## Create membranes

In [ ]:
# ----------------------------------

# CONFIGURATION

# ----------------------------------



NUMBER_OF_PROTEINS = {
    "avg_membrane": [10, 40, 80, 100, 110, 117]
}

ANGLES = range(360)
N = 10

# -----------------------------------

# LOGGING

# -----------------------------------

logging.basicConfig(
    format="%(asctime)s | %(levelname)s | %(message)s", level=logging.INFO
)

# --------------------------------

# UTILITY FUNCTIONS

# --------------------------------


def ensure_output_dir(out_dir: Path) -> None:
    """Ensure output directory exists"""
    out_dir.mkdir(parents=True, exist_ok=True)


def generate_rotated_polygons(
    n: int, base_polygon: Polygon, angles: range
) -> list[Polygon]:
    """Generate n rotated copies of a base polygon"""
    return [rotate(base_polygon, random.choice(angles)) for _ in range(n)]


def generate_roated_polygons_membrane(n_cytb6f: int, angles: range) -> list[Polygon]:
    return (
        [
            rotate(proteins["3JCU-PSII-spinach"]["polygon"][0], random.choice(ANGLES))
            for _ in range(int(n_cytb6f * 2.6))
        ]
        + [
            rotate(
                proteins["6RQF-Cyt-spinach"]["polygon"][0],
                random.choice(ANGLES),
            )
            for _ in range(int(n_cytb6f))
        ]
        + [
            rotate(proteins["7E0K-LHCII-chlamy"]["polygon"][0], random.choice(ANGLES))
            for _ in range(int(n_cytb6f * 14.1))
        ]
    )


def save_polygons_to_wkt(file_path: Path, polygons: list[Polygon]) -> None:
    """Save polygons as WKT to a file"""
    with file_path.open("w") as f:
        for poly in polygons:
            f.write(poly.wkt + "\n")


# --------------------------------

# MAIN WORKER FUNCTION

# --------------------------------


def process_one_case(args: tuple[str, int, int, str]) -> Path:
    polygon_key, nprot, i = args

    # Polygon source and output directory
    if polygon_key != "avg_membrane":
        base_polygon = proteins[polygon_key]["polygon"][0]
    out_dir = OUT / polygon_key
    ensure_output_dir(out_dir)

    file_path = Path(out_dir / f"polygons_{polygon_key}_{nprot}-{i}.wkt")

    if file_path.exists():
        logging.info(f"{file_path.name} already exists, skipping.")
        return file_path

    if polygon_key == "avg_membrane":
        polygons = generate_roated_polygons_membrane(nprot, ANGLES)
    else:
        polygons = generate_rotated_polygons(nprot, base_polygon, ANGLES)
    world, placed_polygons, not_placed, ghosts = cm.geo_utils.placement_routine(
        polygons,
        dimensions=[5000, 5000],
        step_size=15,
        rotation_angle=360,
        max_iter_placement=10000,
    )

    extended_world = world + list(np.array(ghosts).ravel())
    save_polygons_to_wkt(file_path, extended_world)
    logging.info(f"Saved {len(extended_world)} polygons to {file_path}")

    return file_path


tasks = [
    (polygon_key, nprot, i)
    for polygon_key, nprot_list in NUMBER_OF_PROTEINS.items()
    for nprot in nprot_list
    for i in range(N)
]

with Pool(processes=N_PROCESSES) as pool:
    results = pool.map(process_one_case, tasks)

logging.info(
    f"Completed processing {len(results)} tasks"
    f"across {len(NUMBER_OF_PROTEINS)} polygon types"
)

## Plot membrane examples

In [ ]:
NUMBER_OF_PROTEINS = {
    "avg_membrane": [10, 40, 80, 100, 110, 117]
}


fig, axes = plt.subplots(2,3, figsize = (12, 8))
axes_flat = axes.flatten()

for idx, i in enumerate(NUMBER_OF_PROTEINS["avg_membrane"]):
    plot_membrane(f"output/avg_membrane/polygons_avg_membrane_{i}-0.wkt", ax=axes_flat[idx])

In [ ]:
rec = box(0, 0, 5000, 5000)

for i in NUMBER_OF_PROTEINS["avg_membrane"]:
    file_map = (OUT/"avg_membrane").glob(f"*_{i}-*")
    print(f"\nFiles for Cytb6f {i}")
    for idx, f in enumerate(file_map):
        p117 = cm.geo_utils.readwkt(f)
        inter = rec.intersection(unary_union(p117))
        fraction = inter.area/ rec.area

        print(f"{idx}. file: {f.name} fraction: {fraction}")

        

## Diffusion

In [ ]:
cfg = cm.brownian_lattice.ExperimentLatticeConfig()
cfg.replicates = 3000
cfg.diff_coefficient = 3.5e9  # Å²/s == 3.5×10⁻⁷ cm²/s
cfg.particle_radius = 5
cfg.dimensions = (0, 5000)
cfg.random_start = True
cfg.has_ghost = True
cfg.store_history = False
cfg.workers = 5
cfg.nsteps = 1_000_000
cfg.save_every = 1000

fig, ax = plt.subplots()

for i in NUMBER_OF_PROTEINS["avg_membrane"]:
    file_lst = list((OUT/"avg_membrane").glob(f"*_{i}-*"))
    ens_exp = cm.brownian_lattice.EnsembleExperimentLattice(file_lst, cfg)
    run = ens_exp.run()
    coverage = run.get_mean_coverage() * 100

    run_dif = run.get_mean_diff_coefficients_dist()
    norm_diff_dist = run_dif["D_dist"] / (4 * cfg.diff_coefficient)
    norm_diff_dist_ci_low = run_dif["D_dist_ci_low"] / (4 * cfg.diff_coefficient)
    norm_diff_dist_ci_high = run_dif["D_dist_ci_high"] / (
        4 * cfg.diff_coefficient
    )
    norm_diff_dist.iloc[0] = 1.0
    sqrt_msd = np.sqrt(run_dif["MSD"])

    
    ax.plot(
        sqrt_msd,
        norm_diff_dist,
        label=f"r = {coverage:.0f}%",
        linewidth=2,
        alpha=0.7,)

    ax.fill_between(
            sqrt_msd,
            norm_diff_dist_ci_low,
            norm_diff_dist_ci_high,
            alpha=0.3,
            color=ax.lines[-1].get_color(),
        )

ax.set_xlabel(r"$\langle r \rangle$ (Å)", fontsize=12)
ax.set_ylabel(r"$D(r) / D_0$", fontsize=12)
ax.tick_params(labelsize=10)
ax.grid(True, linestyle=":", linewidth=0.5, alpha=0.5)  # noqa: FBT003
ax.legend(loc="lower right")
ax.set_xlim(0, 350)
    

In [ ]:
cfg = cm.brownian_lattice.ExperimentLatticeConfig()
cfg.replicates = 10
cfg.diff_coefficient = 3.5e9  # Å²/s == 3.5×10⁻⁷ cm²/s
cfg.particle_radius = 5
cfg.dimensions = (0, 5000)
cfg.random_start = True
cfg.has_ghost = True
cfg.store_history = True
cfg.workers = 5
cfg.nsteps = 1_000
cfg.save_every = 10

p = cm.geo_utils.readwkt(OUT/"avg_membrane/polygons_avg_membrane_117-8.wkt")
cfg.shapes = p

exp = cm.brownian_lattice.ExperimentLattice(cfg)
run = exp.run()

In [ ]:
plt.imshow(exp.kernel)

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# from celluloid import Camera
# from copy import deepcopy

# particle = deepcopy(exp.kernel)
# r = particle.shape[0] // 2
# H, W = exp.raster.shape

# positions = run.trajectories[0].data[["Y", "X"]].to_numpy()

# fig, ax = plt.subplots()
# camera = Camera(fig)

# for cy, cx in positions:
#     grid = np.zeros((H, W))

#     gy0, gy1 = max(cy-r, 0), min(cy+r+1, H)
#     gx0, gx1 = max(cx-r, 0), min(cx+r+1, W)
#     py0, py1 = gy0-(cy-r), gy1-(cy-r)
#     px0, px1 = gx0-(cx-r), gx1-(cx-r)

#     grid[gy0:gy1, gx0:gx1][particle[py0:py1, px0:px1]] = 1

#     overlay = exp.raster + grid  # overlap shows as brighter

#     ax.imshow(overlay, cmap="gray")
#     ax.set_xlim(cx - 50, cx + 50)
#     ax.set_ylim(cy + 50, cy - 50)
#     camera.snap()

# camera.animate().save("particle.gif", writer="pillow")